# FICTURE F1

Each transcript gets two cell-type labels: the FICTURE factor of the nearest pixel (≤5 µm) and the cell type of the segmented cell it lies in. F1 is computed per cell type; macro = mean over cell types, vascular = ECs, Pericytes, SMCs, VLMCs. 3D methods are skipped.

In [ ]:
from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.spatial as ss
import seaborn as sns
import spatialdata as sd
from matplotlib import font_manager
from scipy.cluster.hierarchy import leaves_list, linkage
from cellseg_benchmark import BASE_PATH
from cellseg_benchmark import ficture_utils as fu
from cellseg_benchmark._constants import method_colors, true_cluster
from cellseg_benchmark.metrics import (
    clean_method_name,
    compute_ficture_f1,
    compute_metric_for_all_methods,
    get_negative_markers,
    get_positive_markers,
)
from cellseg_benchmark.metrics.ficture import plot_ficture_f1, plot_ficture_f1_heatmap

font_manager.fontManager.addfont(f"{BASE_PATH}/misc/Arial.ttf")
plt.rcParams["font.family"] = "Arial"
cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
metrics = Path(BASE_PATH) / "metrics"
out = metrics / "merged/ficture"
factors = {c: sorted(Path(BASE_PATH).glob(f"samples/{c}_*/results/Ficture/ficture_transcript_factors.parquet")) for c in cohorts}

## Compute

In [ ]:
%%time
for c in cohorts:
    compute_metric_for_all_methods(compute_ficture_f1, c, "ficture/ficture_f1.csv", skip_3D=True)

## Per method

In [ ]:
plot_ficture_f1("merged")
for c in cohorts:
    plot_ficture_f1(c)
    plot_ficture_f1(c, "f1_macro_vascular")

## Per cell type

In [ ]:
for c in cohorts:
    plot_ficture_f1_heatmap(c)

## Agreement with marker F1

Marker F1 scores cell types from expression, FICTURE F1 from spatial factor identity. A high rank correlation means both metrics order the methods similarly.

In [ ]:
mf1 = pd.concat([pd.read_csv(metrics / c / "marker_gene_metrics/marker_f1_score_cell_type_revised.csv", index_col=0).assign(cohort=c) for c in cohorts])
ff1 = pd.concat([pd.read_csv(metrics / c / "ficture/ficture_f1.csv", index_col=0).assign(cohort=c) for c in cohorts])
both = pd.concat([
    mf1.groupby(["cohort", "method", "sample"]).f1_score.mean().groupby(["cohort", "method"]).mean().rename("marker"),
    ff1.query("sample != 'all_samples' and cell_type == 'f1_macro_all'").groupby(["cohort", "method"]).f1.mean().rename("ficture"),
], axis=1).dropna().reset_index()

fig, axs = plt.subplots(1, len(cohorts), figsize=(5 * len(cohorts), 4.5))
for ax, c in zip(axs, cohorts):
    g = both[both.cohort == c]
    xs = np.array([g.marker.min(), g.marker.max()])
    ax.plot(xs, np.polyval(np.polyfit(g.marker, g.ficture, 1), xs), color="0.8", lw=1.5, ls="--", zorder=0)
    ax.scatter(g.marker, g.ficture, c=g.method.map(lambda m: method_colors.get(m, "0.6")), s=30)
    for _, r in g.iterrows():
        ax.annotate(clean_method_name(r.method), (r.marker, r.ficture), fontsize=5)
    rho = g[["marker", "ficture"]].corr("spearman").iloc[0, 1]
    ax.set(title=f"{c} (Spearman ρ = {rho:.2f})", xlabel="Marker F1", ylabel="FICTURE macro F1")
    sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(out / "plots/ficture_vs_marker_f1.png", dpi=300, bbox_inches="tight")

## Is FICTURE a clean reference?

FICTURE is initialised from the ABC Atlas pseudobulk, so its factors are the reference cell-type profiles projected onto pixels, without segmentation. Segmented cells are annotated against the same reference, so FICTURE F1 isolates errors introduced by segmentation. Two checks that the pixel labels are biologically sound:

- **Markers:** each factor should be enriched for its cell type's positive markers (consistency check, circular by construction) and depleted of its negative markers.
- **Vascular architecture:** without any expression input, mural cells (Pericytes, SMCs) should sit next to ECs, and SMCs next to VLMCs around large vessels.

### Markers

In [ ]:
%%time
n_samples = None
counts = {}
for c in cohorts:
    per_sample = []
    for p in factors[c][:n_samples]:
        s = p.parents[2].name
        t = pd.read_parquet(p)
        pts = sd.read_zarr(Path(BASE_PATH) / "samples" / s / "sdata_z3.zarr", selection=("points",))[f"{s}_transcripts"]
        pts = pts[["x", "gene"]].compute()
        assert len(pts) == len(t) and np.allclose(pts["x"].to_numpy(np.float32), t["x"].to_numpy())
        tab = pd.DataFrame({"factor": t["factor"].to_numpy(), "gene": pts["gene"].to_numpy()}).groupby(["factor", "gene"], observed=True).size().unstack(fill_value=0)
        per_sample.append(tab.groupby(tab.index.map(fu.factor_cell_types(fu.find_ficture_output(s, BASE_PATH)))).sum())
        del t, pts
    counts[c] = pd.concat(per_sample).fillna(0).groupby(level=0).sum()
    print(c, len(per_sample), "samples,", f"{int(counts[c].values.sum()):,} transcripts")

In [ ]:
owner, types = {}, {}
fig, axs = plt.subplots(1, len(cohorts), figsize=(len(cohorts) * 4.5, 4.5))
for ax, c in zip(axs, cohorts):
    pos = get_positive_markers(subset_genes=set(counts[c].columns))
    m = pd.DataFrame([(g, true_cluster.get(t, t)) for t, genes in pos.items() for g in genes], columns=["gene", "type"]).drop_duplicates()
    owner[c] = m[~m.gene.duplicated(keep=False) & m.type.notna()].set_index("gene").type
    types[c] = sorted(set(owner[c]) & set(counts[c].index))
    conf = counts[c][owner[c].index].T.groupby(owner[c]).sum().reindex(index=types[c], columns=types[c], fill_value=0)
    mix = conf.div(conf.sum(axis=1), axis=0)
    sns.heatmap(mix, cmap="Blues", vmin=0, vmax=1, annot=mix.apply(lambda col: col.map(lambda v: f"{v:.2f}".lstrip("0"))), fmt="",
                annot_kws={"size": 8}, cbar=False, square=True, ax=ax)
    ax.set(title=c, xlabel="FICTURE factor", ylabel="Marker genes of" if ax is axs[0] else "")
fig.suptitle("Where each cell type's marker transcripts end up")
fig.tight_layout()
fig.savefig(out / "plots/ficture_reference_marker_mix.png", dpi=300, bbox_inches="tight")

In [ ]:
enrich = {}
for c in cohorts:
    neg, _ = get_negative_markers(c)
    neg = neg.groupby(neg.index.map(lambda t: true_cluster.get(t, t))).all()
    share = counts[c].div(counts[c].sum(axis=1), axis=0)
    overall = counts[c].sum() / counts[c].values.sum()
    enrich[c] = np.log2(pd.DataFrame({
        t: {"positive": share.loc[t, pos_genes].sum() / overall[pos_genes].sum(),
            "negative": share.loc[t, neg_genes].sum() / overall[neg_genes].sum()}
        for t in types[c] if t in neg.index
        for pos_genes, neg_genes in [(owner[c].index[owner[c] == t], neg.columns[neg.loc[t]].intersection(counts[c].columns))]
    }).T)
enrich = pd.concat(enrich, names=["cohort", "cell_type"])
rows = enrich.groupby("cell_type").positive.mean().sort_values().index
y = pd.Series(range(len(rows)), index=rows)

fig, ax = plt.subplots(figsize=(5, 0.4 * len(rows) + 1))
for (c, d), marker in zip(enrich.groupby("cohort", sort=False), "os^"):
    d = d.droplevel("cohort")
    ax.hlines(y[d.index], d.negative, d.positive, color="0.9", lw=1.5, zorder=1)
    ax.scatter(d.positive, y[d.index], s=30, marker=marker, color="#3b6ea5", zorder=3)
    ax.scatter(d.negative, y[d.index], s=30, marker=marker, color="#c44e52", zorder=3)
    ax.scatter([], [], s=30, marker=marker, color="0.5", label=c)
ax.axvline(0, color="0.5", lw=0.8, ls="--")
ax.set_yticks(y, rows)
ax.set(xlabel="log2 marker share in factor (observed / expected)\nred: negative markers, blue: positive markers")
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, fontsize=8, title=None)
sns.despine(ax=ax, left=True)
fig.savefig(out / "plots/ficture_reference_marker_enrichment.png", dpi=300, bbox_inches="tight")
enrich.to_csv(out / "ficture_reference_marker_enrichment.csv")
enrich.groupby("cell_type").agg(["mean", "std"]).round(2)

### Vascular architecture

In [ ]:
types_all = sorted({t for t in true_cluster.values() if t})
n = len(types_all)


def enrichment(xy, lab, r, k=20):
    """log2 observed/expected label pairs among neighbours within r µm; self-pairs excluded, chance = 0."""
    idx = ss.cKDTree(xy).query(xy, k=k, distance_upper_bound=r)[1][:, 1:]
    ok = idx < len(xy)
    code = pd.Categorical(lab, categories=types_all).codes.astype(np.int64)
    a, b = np.repeat(code, ok.sum(axis=1)), code[idx[ok]]
    keep = (a >= 0) & (b >= 0)
    p = np.bincount(a[keep] * n + b[keep], minlength=n * n).reshape(n, n)
    f = p.sum(axis=1) / p.sum()
    np.fill_diagonal(p, 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        e = np.log2(p / (p.sum() * np.outer(f, f) / (1 - (f**2).sum())))
    return pd.DataFrame(np.where(np.isinf(e), np.nan, e), types_all, types_all)


# FICTURE: factor at each transcript (1M per sample), segmentation-free
pix = {}
for p in [p for c in cohorts for p in factors[c]]:
    t = pd.read_parquet(p)
    t = t.sample(min(1_000_000, len(t)), random_state=0)
    lab = t.factor.map(fu.factor_cell_types(fu.find_ficture_output(p.parents[2].name, BASE_PATH))).to_numpy()
    pix[p.parents[2].name] = enrichment(t[["x", "y"]].to_numpy(), lab, 10)

# orthogonal view: annotated cells of one segmentation (centroids)
cell = {}
for c in cohorts:
    a = ad.read_h5ad(Path(BASE_PATH) / "analysis" / c / "Cellpose_1_nuclei_model/adatas/adata_integrated.h5ad.gz", backed="r")
    obs = a.obs.assign(x=a.obsm["spatial_microns"][:, 0], y=a.obsm["spatial_microns"][:, 1])
    cell |= {s: enrichment(d[["x", "y"]].to_numpy(), d.cell_type_revised.astype(str).map(true_cluster).to_numpy(), 20)
             for s, d in obs.groupby("sample", observed=True) if s in pix}

mean = pd.concat(pix.values()).groupby(level=0).mean().loc[types_all, types_all]
std = pd.concat(pix.values()).groupby(level=0).std().loc[types_all, types_all]
cell_mean = pd.concat(cell.values()).groupby(level=0).mean().loc[types_all, types_all]
leaves = leaves_list(linkage(mean.fillna(0), "average"))
offdiag = ~np.eye(n, dtype=bool)
rho = pd.Series(mean.values[offdiag]).corr(pd.Series(cell_mean.values[offdiag]), method="spearman")

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, (title, m) in zip(axs, [(f"FICTURE pixels, {len(pix)} samples, {len(cohorts)} cohorts (≤10 µm)", mean), ("Cellpose1 N cells (≤20 µm)", cell_mean)]):
    m = m.iloc[leaves, leaves]
    sns.heatmap(m, mask=np.eye(n, dtype=bool), cmap="RdBu_r", center=0, vmin=-2, vmax=2, square=True,
                annot=m.apply(lambda col: col.map(lambda v: f"{v:.1f}")), fmt="", annot_kws={"size": 7},
                cbar_kws={"label": "log2 neighbour enrichment", "shrink": 0.5}, ax=ax)
    ax.set(xlabel="", ylabel="", title=title)
    ax.tick_params(length=0)
fig.suptitle(f"Neighbour enrichment (Spearman ρ pixels vs cells = {rho:.2f})")
fig.tight_layout()
fig.savefig(out / "plots/ficture_reference_neighbours.png", dpi=300, bbox_inches="tight")
mean.to_csv(out / "ficture_reference_neighbours.csv")

for u, v in [("ECs", "Pericytes"), ("ECs", "SMCs"), ("ECs", "VLMCs"), ("SMCs", "VLMCs"), ("BAMs", "SMCs"), ("Pericytes", "SMCs")]:
    print(f"{u}–{v}: {mean.loc[u, v]:.2f} ± {std.loc[u, v]:.2f} (cells {cell_mean.loc[u, v]:.2f})")